# [초급 프로젝트] 4팀_김명환

---
---

## kaggle에 보고된 값을 비교하자

# 프로그래밍

In [3]:
# 기본 라이브러리 (중복 제거 및 정리)

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, average_precision_score

# --- 이미지 처리 ---
import cv2
from PIL import Image, ImageFilter, ImageDraw
import albumentations as A

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
# 문제 있는 v2 import 제거하고 필요시에만 개별적으로 import
# from torchvision.transforms import v2, functional as TF
from torchvision.transforms import functional as TF
from torchvision.datasets import CocoDetection
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- COCO 데이터셋 관련 ---
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask

# --- 딥러닝 모델 ---
import timm

# --- 기본 라이브러리 ---
import os
import sys
import re
import csv
import copy
import json
import math
import random
import yaml
import shutil
import requests
import xml.etree.ElementTree as ET
from pathlib import Path

# --- 데이터 분석 및 시각화 ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- 시간 관련 ---
from datetime import datetime, timezone, timedelta
import pytz

# --- 진행률 표시 ---
import IPython.display
from tqdm.notebook import tqdm

# --- 시간대 설정 ---
__kst = pytz.timezone('Asia/Seoul')

# --- GPU 설정 ---
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

# --- 재현 가능한 결과를 위한 시드 설정 ---
np.random.seed(42)
torch.manual_seed(42)
if __device.type == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치: {__device}")

라이브러리 로드 완료 사용장치: cpu


In [4]:
from urllib.request import urlretrieve; urlretrieve("https://raw.githubusercontent.com/c0z0c/jupyter_hangul/refs/heads/beta/helper_c0z0c_dev.py", "helper_c0z0c_dev.py")
import importlib
import helper_c0z0c_dev as helper
importlib.reload(helper)

🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환


<module 'helper_c0z0c_dev' from 'd:\\GoogleDrive\\codeit_ai_health_eat\\scripts\\김명환\\helper_c0z0c_dev.py'>

In [5]:
import os, sys
from pathlib import Path

utils_dir = None
if helper.is_colab:
    utils_dir = "/content/drive/MyDrive/codeit_ai_health_eat/src/python_modules/utils"
else:
    utils_dir = os.path.join(Path.cwd().drive + '\\', 'GoogleDrive', "codeit_ai_health_eat", "src", "python_modules", "utils")

print("utils_dir:", utils_dir)

sys.path.append(str(utils_dir))
print("sys.path:", sys.path)
import importlib
import health_ea_utils as heu
importlib.reload(heu)
from health_ea_utils import *

print("helper.__file__:", helper.__file__)
print("health_ea_utils.__file__:", heu.__file__)


utils_dir: d:\GoogleDrive\codeit_ai_health_eat\src\python_modules\utils
sys.path: ['c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\python310.zip', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\DLLs', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\lib', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827', '', 'C:\\Users\\sw1\\AppData\\Roaming\\Python\\Python310\\site-packages', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\lib\\site-packages', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\lib\\site-packages\\win32', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\lib\\site-packages\\win32\\lib', 'c:\\Users\\sw1\\anaconda3\\envs\\env_colab_250827\\lib\\site-packages\\Pythonwin', 'd:\\GoogleDrive\\codeit_ai_health_eat\\src\\python_modules\\utils']
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.1.3 (v2.x+): 호환성 모드 적용됨
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = d:\GoogleDrive\codeit_ai_health_eat\scripts\김명환
🌐 https://c0z0c.github.io/ju

In [7]:
def image_result_sample(self, result_json):
    import json
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from PIL import Image
    import base64
    from io import BytesIO

    # JSON 문자열 파싱
    result = json.loads(result_json)
    bboxs = result['bboxs']
    img_path = result.get('img_path', None)

    # 원본 이미지 로드
    if img_path is not None:
        org_img = Image.open(img_path).convert("RGB")
    else:
        org_img = None

    n = len(bboxs)
    fig = plt.figure(figsize=(4, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[2, 1])

    # 1행: 원본 이미지 + 박스
    ax1 = fig.add_subplot(gs[0])
    if org_img is not None:
        ax1.imshow(org_img)
        for box in bboxs:
            xyxy = box['xyxy']
            class_name = box['class_name']
            class_score = box['class_score']
            drug_N = box['drug_info']['drug_N']
            dl_name = box['drug_info']['dl_name']
            x1, y1, x2, y2 = xyxy
            w, h = x2-x1, y2-y1
            rect = mpatches.Rectangle((int(x1), int(y1)), int(w), int(h),
                                    linewidth=2, edgecolor='red', facecolor='none')
            ax1.add_patch(rect)
            label = f"{class_name} {class_score:.2f}\n{drug_N}\n{dl_name}"
            ax1.text(x1, y1-5, label, color='red', fontsize=10, backgroundcolor='white', alpha=0.8)
        ax1.axis('off')
        ax1.set_title('이미지 + 박스', pad=5)
    else:
        ax1.set_title('원본 이미지 없음')
        ax1.axis('off')

    # 2행: 각 박스 Crop 이미지 한 행에 나란히
    ax2 = fig.add_subplot(gs[1])
    crop_imgs = []
    for box in bboxs:
        # base64 이미지를 PIL로 변환
        img_b64 = box['img']
        img_bytes = base64.b64decode(img_b64)
        crop_img = Image.open(BytesIO(img_bytes)).convert("RGB")
        crop_imgs.append(np.array(crop_img))

    if n > 0:
        heights = [img.shape[0] for img in crop_imgs]
        max_h = max(heights)
        resized_imgs = []
        crop_positions = []
        x_offset = 0
        for i, img in enumerate(crop_imgs):
            if img.shape[0] != max_h:
                pil_img = Image.fromarray(img)
                ratio = max_h / img.shape[0]
                new_w = int(img.shape[1] * ratio)
                pil_img = pil_img.resize((new_w, max_h))
                img = np.array(pil_img)
            resized_imgs.append(img)
            crop_positions.append((x_offset, img.shape[1]))
            x_offset += img.shape[1]
        concat_img = np.concatenate(resized_imgs, axis=1)
        ax2.imshow(concat_img)
        ax2.axis('off')
        ax2.set_title('알약 이미지들', pad=5)
        # 각 crop 이미지 위에 라벨 출력
        for i, (start_x, width) in enumerate(crop_positions):
            class_name = bboxs[i]['class_name']
            class_score = bboxs[i]['class_score']
            label = f"{class_name}\n{class_score:.2f}"
            ax2.text(start_x + width // 2, 10, label, color='red', fontsize=10,
                    backgroundcolor='white', ha='center', va='top', alpha=0.8)
    else:
        ax2.set_title('알약 없음')
        ax2.axis('off')

    # 여백 최소화
    plt.subplots_adjust(hspace=0, top=1, bottom=0)
    plt.tight_layout(pad=0)
    buf = BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    pil_img = Image.open(buf)
    return pil_img


In [8]:
# diff_result(test_images_path, result_a_csv_path, result_b_csv_path, output_folder)
# annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
# 1,1,16550,558.0531616210938,75.6333999633789,398.298583984375,396.9631881713867,0.9944773316383361
# test_images_path에 image_id 이름으로 원본파일이 있다.
# image_id 위에  bbox_x, bbox_y, bbox_w, bbox_h 를그리고 category_id, score를 표시한다
# result_a는 파란색 result_b는 빨간색으로 표시한다
# image_result_sample를 참고하여 image_result를 만든다 원본이미지위에 박스와정보, 그밑에 crop이미지들을 나란히
# diff_result에서 이용한다.


In [9]:
def image_result(img_path, boxes_a, boxes_b, title="비교 결과"):
    """
    image_result_sample을 참고하여 만든 비교용 이미지 생성 함수
    
    Args:
        img_path: 원본 이미지 경로
        boxes_a: result_a의 박스 리스트 (파란색)
        boxes_b: result_b의 박스 리스트 (빨간색)
        title: 이미지 제목
    
    Returns:
        PIL Image: 비교 결과 이미지
    """
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from PIL import Image
    from io import BytesIO
    
    # 원본 이미지 로드
    if os.path.exists(img_path):
        org_img = Image.open(img_path).convert("RGB")
    else:
        print(f"이미지를 찾을 수 없습니다: {img_path}")
        return None
    
    # 모든 박스들 합치기 (구분을 위해 색상 정보 추가)
    all_boxes = []
    for box in boxes_a:
        box_info = box.copy()
        box_info['color'] = 'blue'
        box_info['label_prefix'] = 'A:'
        all_boxes.append(box_info)
    
    for box in boxes_b:
        box_info = box.copy()
        box_info['color'] = 'red' 
        box_info['label_prefix'] = 'B:'
        all_boxes.append(box_info)
    
    n_total = len(all_boxes)
    
    if n_total == 0:
        # 박스가 없는 경우 원본 이미지만 반환
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.imshow(org_img)
        ax.set_title(f'{title} - 검출 결과 없음')
        ax.axis('off')
        
        buf = BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
        plt.close(fig)
        buf.seek(0)
        result_img = Image.open(buf).copy()
        buf.close()
        return result_img
    
    # 레이아웃 설정 (상단: 원본+박스, 하단: crop 이미지들)
    fig = plt.figure(figsize=(max(12, n_total * 2), 10))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.1)
    
    # 1행: 원본 이미지 + 박스들
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(org_img)
    
    # 박스 그리기
    for box in all_boxes:
        x, y, w, h = box['bbox_x'], box['bbox_y'], box['bbox_w'], box['bbox_h']
        category_id = box['category_id']
        score = box['score']
        color = box['color']
        prefix = box['label_prefix']
        
        # 색상 설정
        edge_color = 'blue' if color == 'blue' else 'red'
        
        # 박스 그리기
        rect = mpatches.Rectangle((x, y), w, h,
                                linewidth=2, edgecolor=edge_color, 
                                facecolor='none', alpha=0.8)
        ax1.add_patch(rect)
        
        # 라벨 텍스트
        label = f"{prefix}{category_id}\n{score:.3f}"
        
        # 텍스트 배경색 설정
        bg_color = 'lightblue' if color == 'blue' else 'lightcoral'
        text_color = 'darkblue' if color == 'blue' else 'darkred'
        
        # 라벨 위치 설정: 파란색은 오른쪽 아래, 빨간색은 왼쪽 위
        if color == 'blue':
            # 파란색: 박스 오른쪽 아래
            label_x, label_y = x + w, y + h + 5
            va = 'top'
            ha = 'right'
        else:
            # 빨간색: 박스 왼쪽 위 (기존 위치)
            label_x, label_y = x, y - 5
            va = 'bottom'
            ha = 'left'
        
        ax1.text(label_x, label_y, label, color=text_color, fontsize=9, 
                backgroundcolor=bg_color, alpha=0.9, 
                verticalalignment=va, horizontalalignment=ha)

    ax1.axis('off')
    ax1.set_title(f'{title} - A(파란색) vs B(빨간색)', pad=10, fontsize=14)
    
    # 2행: Crop 이미지들
    ax2 = fig.add_subplot(gs[1])
    
    if n_total > 0:
        crop_imgs = []
        crop_labels = []
        
        # 각 박스별로 crop 이미지 생성
        for box in all_boxes:
            x, y, w, h = box['bbox_x'], box['bbox_y'], box['bbox_w'], box['bbox_h']
            
            # 좌표 정수 변환 및 이미지 범위 클리핑
            x1, y1 = max(0, int(x)), max(0, int(y))
            x2, y2 = min(org_img.width, int(x + w)), min(org_img.height, int(y + h))
            
            if x2 > x1 and y2 > y1:
                crop_img = org_img.crop((x1, y1, x2, y2))
                crop_imgs.append(np.array(crop_img))
                
                # 라벨 생성 - A/B 구분에 따른 색상 설정
                prefix = box['label_prefix']
                category_id = box['category_id']
                score = box['score']
                color = box['color']
                
                # A/B에 따른 색상 및 배경색 설정
                if color == 'blue':  # A 결과
                    text_color = 'white'
                    bg_color = 'blue'
                else:  # B 결과
                    text_color = 'white'
                    bg_color = 'red'
                
                crop_labels.append({
                    'text': f"{prefix}{category_id}\n{score:.3f}",
                    'text_color': text_color,
                    'bg_color': bg_color,
                    'label_type': 'A' if color == 'blue' else 'B'
                })
        
        if crop_imgs:
            # 높이를 통일하여 가로로 연결
            heights = [img.shape[0] for img in crop_imgs]
            max_h = max(heights)
            
            resized_imgs = []
            crop_positions = []
            x_offset = 0
            
            for i, img in enumerate(crop_imgs):
                if img.shape[0] != max_h:
                    # 비율 유지하며 리사이즈
                    pil_img = Image.fromarray(img)
                    ratio = max_h / img.shape[0]
                    new_w = int(img.shape[1] * ratio)
                    pil_img = pil_img.resize((new_w, max_h), Image.Resampling.LANCZOS)
                    img = np.array(pil_img)
                
                resized_imgs.append(img)
                crop_positions.append((x_offset, img.shape[1]))
                x_offset += img.shape[1]
            
            # 가로로 연결
            concat_img = np.concatenate(resized_imgs, axis=1)
            ax2.imshow(concat_img)
            
            # 각 crop 이미지 위에 라벨 표시 - 색상 구분 강화
            for i, (start_x, width) in enumerate(crop_positions):
                if i < len(crop_labels):
                    label_info = crop_labels[i]
                    
                    # 라벨 텍스트 표시 (상단 중앙)
                    ax2.text(start_x + width // 2, 10, label_info['text'], 
                            color=label_info['text_color'], fontsize=9, weight='bold',
                            backgroundcolor=label_info['bg_color'], ha='center', va='top', 
                            alpha=0.9, bbox=dict(boxstyle="round,pad=0.3", 
                                                facecolor=label_info['bg_color'], 
                                                alpha=0.9))
                    
                    # 추가: 이미지 하단에 A/B 구분 표시
                    label_type = label_info['label_type']
                    ax2.text(start_x + width // 2, max_h - 5, f"[{label_type}]", 
                            color=label_info['text_color'], fontsize=10, weight='bold',
                            backgroundcolor=label_info['bg_color'], ha='center', va='bottom', 
                            alpha=0.9, bbox=dict(boxstyle="round,pad=0.2", 
                                                facecolor=label_info['bg_color'], 
                                                alpha=0.9))
            
            ax2.set_title('검출된 객체들 - A(파란색) vs B(빨간색)', pad=5, fontsize=12)
        else:
            ax2.text(0.5, 0.5, '유효한 crop 이미지 없음', 
                    ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    else:
        ax2.text(0.5, 0.5, '검출 결과 없음', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    
    ax2.axis('off')
    
    # 여백 최소화
    plt.tight_layout(pad=1.0)
    
    # PIL 이미지로 변환
    buf = BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.1, dpi=150)
    plt.close(fig)
    buf.seek(0)
    result_img = Image.open(buf).copy()
    buf.close()
    
    return result_img

In [13]:
def diff_result(test_images_path, result_a_csv_path, result_b_csv_path, output_folder):
    """
    두 CSV 결과를 비교하는 이미지를 생성하는 함수
    
    Args:
        test_images_path: 원본 이미지들이 있는 폴더 경로
        result_a_csv_path: 첫 번째 결과 CSV 파일 (파란색으로 표시)
        result_b_csv_path: 두 번째 결과 CSV 파일 (빨간색으로 표시)
        output_folder: 결과 이미지를 저장할 폴더
    """
    import pandas as pd
    
    # 출력 폴더 생성
    os.makedirs(output_folder, exist_ok=True)
    
    # CSV 파일 읽기
    try:
        df_a = pd.read_csv(result_a_csv_path)
        print(f"Result A loaded: {len(df_a)} detections")
    except Exception as e:
        print(f"Error loading result A: {e}")
        return
    
    try:
        df_b = pd.read_csv(result_b_csv_path)
        print(f"Result B loaded: {len(df_b)} detections")
    except Exception as e:
        print(f"Error loading result B: {e}")
        return
    
    # image_id별로 그룹화
    grouped_a = df_a.groupby('image_id') if len(df_a) > 0 else {}
    grouped_b = df_b.groupby('image_id') if len(df_b) > 0 else {}
    
    # 모든 image_id 수집
    all_image_ids = set()
    if len(df_a) > 0:
        all_image_ids.update(df_a['image_id'].unique())
    if len(df_b) > 0:
        all_image_ids.update(df_b['image_id'].unique())
    
    print(f"Processing {len(all_image_ids)} unique images...")
    
    # 각 이미지별로 처리
    success_count = 0
    error_count = 0
    
    all_image_ids = [615]
    
    # print(f"df_a에서 image_id={all_image_ids}")
    # print(df_a[df_a['image_id'].isin(all_image_ids)])

    # print("df_b에서 image_id=={all_image_ids}")
    # print(df_b[df_b['image_id'].isin(all_image_ids)])
    
    for image_id in tqdm(sorted(all_image_ids), desc="Processing images", **get_tqdm_kwargs()):
        # 이미지 파일 찾기
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
            potential_path = os.path.join(test_images_path, f"{image_id}{ext}")
            if os.path.exists(potential_path):
                img_path = potential_path
                break
        
        if img_path is None:
            print(f"이미지를 찾을 수 없습니다: {image_id}")
            error_count += 1
            continue
        
        # 해당 이미지의 박스들 추출
        boxes_a = []
        boxes_b = []
        
        if hasattr(grouped_a, 'get_group'):
            try:
                group_a = grouped_a.get_group(image_id)
                boxes_a = group_a.to_dict('records')
            except KeyError:
                pass  # 해당 이미지에 대한 결과가 없음
        
        if hasattr(grouped_b, 'get_group'):
            try:
                group_b = grouped_b.get_group(image_id)
                boxes_b = group_b.to_dict('records')
            except KeyError:
                pass  # 해당 이미지에 대한 결과가 없음
        
        # 비교 이미지 생성
        title = f"Image {image_id}"
        result_img = image_result(img_path, boxes_a, boxes_b, title)
        
        if result_img is not None:
            # 결과 저장 (PNG -> JPG로 변경)
            output_path = os.path.join(output_folder, f"{image_id}_comparison.jpg")
            # JPG 저장 시 RGB 모드 확인 및 품질 설정
            if result_img.mode == 'RGBA':
                result_img = result_img.convert('RGB')
            result_img.save(output_path, 'JPEG', quality=95)
            success_count += 1
        else:
            error_count += 1
            
    
    print(f"\n완료: {success_count}개 성공, {error_count}개 실패")
    print(f"결과 저장 폴더: {output_folder}")

In [11]:
# 테스트용 경로 (실제 경로로 변경하세요)
test_images_path = r"D:\dataset\kaggle_code_it_data\ai04-level1-project.zip.unzip\test_images"  # 원본 이미지 폴더
result_a_csv = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\2stage_result_20250918_130256\result_2way_20250918_130256.csv"  # 첫 번째 결과
result_b_csv = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\output_data_resnet101.csv"  # 두 번째 결과
#output_folder = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\2stage_result_20250918_130256_diff"  # 출력 폴더
output_folder = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\2stage_result_20250918_130256_411_diff"  # 출력 폴더

diff_result(test_images_path, result_a_csv, result_b_csv, output_folder)

Result A loaded: 3233 detections
Result B loaded: 3235 detections
Processing 843 unique images...
df_a에서 image_id=[615]
      annotation_id  image_id  category_id  bbox_x  bbox_y  bbox_w  bbox_h  \
2590           2591       615        19231     208     754     239     350   
2591           2592       615        22073     673     821     184     186   
2592           2593       615         3350     393     215     176     174   

       score  
2590  1.0000  
2591  0.9998  
2592  0.9305  
df_b에서 image_id=={all_image_ids}
      annotation_id  image_id  category_id      bbox_x      bbox_y  \
2332           2333       615         3350  388.059998  211.250320   
2333           2334       615        22073  672.931152  819.715942   
2334           2335       615        19231  209.280411  742.179016   
2335           2336       615        18356  209.666931  749.831543   

          bbox_w      bbox_h     score  
2332  185.193604  182.417740  0.988077  
2333  188.193909  188.500061  0.907362  


In [12]:
def diff_result_df(test_images_path, result_a_csv_path, result_b_csv_path, output_folder):
    """
    두 CSV 결과를 비교하는 이미지를 생성하는 함수
    
    Args:
        test_images_path: 원본 이미지들이 있는 폴더 경로
        result_a_csv_path: 첫 번째 결과 CSV 파일 (파란색으로 표시)
        result_b_csv_path: 두 번째 결과 CSV 파일 (빨간색으로 표시)
        output_folder: 결과 이미지를 저장할 폴더
    """
    import pandas as pd
    
    # 출력 폴더 생성
    os.makedirs(output_folder, exist_ok=True)
    
    # CSV 파일 읽기
    try:
        df_a = pd.read_csv(result_a_csv_path)
        print(f"Result A loaded: {len(df_a)} detections")
    except Exception as e:
        print(f"Error loading result A: {e}")
        return
    
    try:
        df_b = pd.read_csv(result_b_csv_path)
        print(f"Result B loaded: {len(df_b)} detections")
    except Exception as e:
        print(f"Error loading result B: {e}")
        return
    
    # df_a, df_b는 이미 로드되어 있다고 가정

    # 1. image_id별로 df_a와 df_b의 category_id 집합 비교
    diff_image_ids = []
    diff_summary = []

    for image_id in sorted(set(df_a['image_id']).union(df_b['image_id'])):
        cats_a = set(df_a[df_a['image_id'] == image_id]['category_id'])
        cats_b = set(df_b[df_b['image_id'] == image_id]['category_id'])
        if cats_a != cats_b:
            diff_image_ids.append(image_id)
            diff_summary.append({
                'image_id': image_id,
                'category_id_a': sorted(list(cats_a)),
                'category_id_b': sorted(list(cats_b)),
                'only_in_a': sorted(list(cats_a - cats_b)),
                'only_in_b': sorted(list(cats_b - cats_a)),
            })

    print(f"category_id가 다르게 존재하는 image_id 개수: {len(diff_image_ids)}")
    for item in diff_summary[:10]:  # 상위 10개만 출력
        print(f"image_id: {item['image_id']}")
        print(f"  df_a category_id: {item['category_id_a']}")
        print(f"  df_b category_id: {item['category_id_b']}")
        print(f"  only_in_a: {item['only_in_a']}")
        print(f"  only_in_b: {item['only_in_b']}")
        print("-" * 40)


# 테스트용 경로 (실제 경로로 변경하세요)
test_images_path = r"D:\dataset\kaggle_code_it_data\ai04-level1-project.zip.unzip\test_images"  # 원본 이미지 폴더
result_a_csv = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\2stage_result_20250918_130256\result_2way_20250918_130256.csv"  # 첫 번째 결과
result_b_csv = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\output_data_resnet101.csv"  # 두 번째 결과
#output_folder = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\2stage_result_20250918_130256_diff"  # 출력 폴더
output_folder = r"D:\GoogleDrive\codeit_ai_health_eat\scripts\김명환\2stage_result_20250918_130256_411_diff"  # 출력 폴더

diff_result_df(test_images_path, result_a_csv, result_b_csv, output_folder)

Result A loaded: 3233 detections
Result B loaded: 3235 detections
category_id가 다르게 존재하는 image_id 개수: 11
image_id: 277
  df_a category_id: [3482, 19860, 34596, 36636]
  df_b category_id: [3482, 19860, 30307, 36636]
  only_in_a: [34596]
  only_in_b: [30307]
----------------------------------------
image_id: 409
  df_a category_id: [1899, 16550, 25366, 33008]
  df_b category_id: [1899, 10223, 16550, 33008]
  only_in_a: [25366]
  only_in_b: [10223]
----------------------------------------
image_id: 411
  df_a category_id: [1899, 16550, 25366, 33008]
  df_b category_id: [1899, 10223, 16550, 33008]
  only_in_a: [25366]
  only_in_b: [10223]
----------------------------------------
image_id: 615
  df_a category_id: [3350, 19231, 22073]
  df_b category_id: [3350, 18356, 19231, 22073]
  only_in_a: []
  only_in_b: [18356]
----------------------------------------
image_id: 697
  df_a category_id: [3350, 18356, 38161]
  df_b category_id: [3350, 18356, 19231, 38161]
  only_in_a: []
  only_in_b: [192